In [ ]:
from optparse import Values
from statistics import correlation

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


In [ ]:
df_customers = pd.read_csv("DataSet/customers.csv")
df_customers.head()

In [ ]:
df_cs = pd.read_csv("DataSet/credit_profiles.csv")
df_trans = pd.read_csv("DataSet/transactions.csv")

In [ ]:
df_cs.head()

In [ ]:
df_trans.shape

Handling Null Values (Annual Income)

In [ ]:
df_customers.describe()

In [ ]:
df_customers.isnull().sum()

In [ ]:
df_customers.annual_income.isna()

In [ ]:
df_customers[df_customers.annual_income.isna()].head()

In [ ]:
df_customers.annual_income.median()

In [ ]:
df_customers[df_customers.occupation == "Artist"].annual_income.median()

In [ ]:
df_customers[df_customers.occupation == "Freelancer"].annual_income.median()

In [ ]:
df_customers[df_customers.occupation == "Data Scientist"].annual_income.median()

In [ ]:
occupation_wise_median  =df_customers.groupby("occupation").annual_income.median()
occupation_wise_median

In [ ]:
occupation_wise_median["Artist"]

In [ ]:
def get_median_val(row):
    if pd.isnull(row["annual_income"]):
        return occupation_wise_median(row["occupation"])
    else:
        return row["annual_income"]

get_median_val({"cust_id" : 3245 ,"occupation" : "Artist" ,"annual_income": 678})

In [ ]:
df_customers["annual_income"] = df_customers.apply(
    lambda row: occupation_wise_median[row["occupation"]] if pd.isnull(row["annual_income"]) else row["annual_income"] ,
    axis =1)

In [ ]:
df_customers.isnull().sum()

In [ ]:
df_customers.iloc[[14,82]]

In [ ]:
plt.figure(figsize=(5,5))
sns.histplot(df_customers["annual_income"],kde = False,color="green",label = 'Data')
plt.title("Histogram of annual income")
plt.show()

DATA CLEANING (Treating Outliers)

In [ ]:
df_customers.describe()

In [ ]:
df_customers[df_customers.annual_income< 100]

In [ ]:
occupation_wise_median

In [ ]:
for index,row in df_customers.iterrows():
    if row['annual_income'] < 100:
        df_customers.at[index,"annual_income"] = occupation_wise_median[row["occupation"]]

In [ ]:
df_customers[df_customers.annual_income<100]

In [ ]:
df_customers.loc[[31,36]]

DATA VISUALISATION (Annual Income)

In [ ]:
avg_income_per_occupation = df_customers.groupby("occupation")["annual_income"].mean()
avg_income_per_occupation

In [ ]:
avg_income_per_occupation.index

In [ ]:
avg_income_per_occupation.values

In [ ]:
sns.barplot(x= avg_income_per_occupation.index, y =avg_income_per_occupation.values,palette="tab10")
plt.xticks(rotation=45)
plt.show()

Treat Oultiers in Age Coloumn

In [ ]:
df_customers.age.describe()

In [ ]:
outliers = df_customers[(df_customers.age>80) | (df_customers.age < 15)]
outliers

In [ ]:
median_age = df_customers.groupby("occupation")["age"].median()
median_age

In [ ]:
for index ,row in outliers.iterrows():
    df_customers.at[index,"age"] = median_age[row["occupation"]]

In [ ]:
df_customers[(df_customers.age<15) | (df_customers.age>80) ]

In [ ]:
df_customers.describe()

Data Visualisation (Age,Gender,Location)

In [ ]:
df_customers.head()

In [ ]:
bin_edges = [17,25,48,65]
bin_labels = ['18-25','26-48','49-65']

df_customers['age_group'] = pd.cut(df_customers["age"],bins=bin_edges, labels=bin_labels)
df_customers.head()

In [ ]:
age_group_counts = df_customers.age_group.value_counts(normalize = True)*100
age_group_counts

In [ ]:
type(age_group_counts)

In [ ]:
plt.pie(
    age_group_counts,
    labels=age_group_counts.index
    ,autopct='%1.1f%%',
    shadow=True,
     explode=(0.1,0.1,0))
plt.show()

In [ ]:
df_customers.location.value_counts()

In [ ]:
df_customers.gender.value_counts()

In [ ]:
customer_location_gender = df_customers.groupby(['location','gender']).size()
customer_location_gender


In [ ]:
customer_location_gender.plot(kind='bar',stacked = True,
                              figsize=(8,4))
plt.title("Customer Distribution by Location and Gender")
plt.show()

Data Cleaning (Credit Score Table)

In [ ]:
df_cs

In [ ]:
df_cs.shape

In [ ]:
df_cs[df_cs['cust_id'].duplicated(keep=False)]

In [ ]:
df_cs_clean1 = df_cs.drop_duplicates(subset='cust_id',keep='last')
df_cs_clean1.shape

In [ ]:
df_cs_clean1[df_cs_clean1['cust_id'].duplicated(keep=False)]

In [ ]:
df_cs_clean1.isnull().sum()

In [ ]:
df_cs_clean1[df_cs_clean1.credit_limit.isnull()]

In [ ]:
df_cs_clean1.credit_limit.value_counts()

In [ ]:
plt.scatter(df_cs_clean1['credit_limit'],df_cs_clean1['credit_score'])
plt.title('Credit Score vs Credit Limit')
plt.xlabel('Credit Limit')
plt.ylabel('Credit Score')
plt.show()

In [ ]:
bin_ranges = [300,450,500,550,600,650,700,750,800]
bin_labels = [f'{start}-{end-1}' for start ,end in zip(bin_ranges,bin_ranges[1:])]
df_cs_clean1['credit_score_range'] = pd.cut(df_cs_clean1['credit_score'],bins=bin_ranges,labels=bin_labels,include_lowest=True,right=False)

df_cs_clean1.head()

In [ ]:
df_cs_clean1[df_cs_clean1.credit_score_range == "700-749"]

In [ ]:
mode_df = df_cs_clean1.groupby("credit_score_range")['credit_limit'].agg(lambda x : x.mode().iloc[0])
mode_df

In [ ]:
df_cs_clean1[df_cs_clean1.credit_limit.isnull()].sample(3)

In [ ]:
df_clean2 =pd.merge(df_cs_clean1,mode_df,on='credit_score_range',suffixes=('',"_mode"))
df_clean2.sample(3)

In [ ]:
df_clean2[df_clean2.credit_limit.isnull()].sample(3)

In [ ]:
df_cs_clean3 = df_clean2.copy()
df_cs_clean3.fillna(df_cs_clean3['credit_limit_mode'],inplace = True)
df_cs_clean3

In [ ]:
df_cs_clean3.shape

In [ ]:
df_cs_clean3.isnull().sum()

In [ ]:
df_cs_clean3.head()

In [ ]:
df_cs_clean3.describe()

In [ ]:
sns.boxplot(df_cs_clean3.outstanding_debt)

In [ ]:
df_cs_clean3[df_cs_clean3.outstanding_debt>df_cs_clean3.credit_limit]

In [ ]:
df_cs_clean3.loc[df_cs_clean3.outstanding_debt>df_cs_clean3.credit_limit, 'outstanding_debt'] = df_cs_clean3['credit_limit']

In [ ]:
df_cs_clean3[df_cs_clean3.outstanding_debt>df_cs_clean3.credit_limit]

Correlation among Credit Profile Variables

In [ ]:
df_cs_clean3.head()

In [ ]:
df_customers.head()

In [ ]:
df_merged = df_customers.merge(df_cs_clean3,on='cust_id',how='inner')
df_merged.head()

In [ ]:
df_merged[['credit_score','credit_limit']].corr()

In [ ]:
numerical_cols = ['credit_score','credit_utilisation','outstanding_debt','credit_limit','annual_income','age']

correlation_matrix = df_merged[numerical_cols].corr()
correlation_matrix

In [ ]:
sns.heatmap(correlation_matrix , annot=True,cmap="YlGnBu",linewidths=0.8)
plt.title('Correlation Matrix')
plt.show()

Transaction Table

In [ ]:
df_trans.describe()

In [ ]:
df_trans.isnull().sum()

In [ ]:
df_trans[df_trans.isnull()]

In [ ]:
df_trans.platform.unique()

In [ ]:
df_trans.platform.value_counts()

In [ ]:
sns.countplot(y = "platform",hue = "product_category",data=df_trans)

In [ ]:
df_trans.platform.fillna(df_trans.platform.mode()[0],inplace = True)

In [ ]:
df_trans.isnull().sum()

Data Cleaning: Treat Outliers using IQR(Transaction Amount)

In [ ]:
df_trans_zero = df_trans[df_trans.tran_amount == 0]
df_trans_zero.shape

In [ ]:
df_trans_zero.head()

In [ ]:
df_trans_zero.product_category.value_counts()